In [ ]:
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

import gc
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm

import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from torch.optim import AdamW

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, f1_score, roc_curve

from transformers import (
    AutoTokenizer,
    AutoModelForMaskedLM,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
    DataCollatorForLanguageModeling
)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

In [ ]:
def set_seed(seed=42):

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [ ]:
VISUALIZE_CLASS_DISTRIBUTION = True
USE_MIXED_PRECISION           = True
USE_LR_SCHEDULER              = True

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
labeled = pd.read_csv("/content/drive/MyDrive/eq_5d/dataset/eq-5d-200-records.csv")
unlabeled = pd.read_csv("/content/drive/MyDrive/eq_5d/dataset/eq-5d-2000-unique-random.csv")

In [ ]:
if VISUALIZE_CLASS_DISTRIBUTION:
    labeled["Label"].value_counts().plot(kind="bar", title="Class Distribution")
    plt.xlabel("Label")
    plt.ylabel("Count")
    plt.tight_layout()
    plt.show()

In [ ]:

labeled["view_1"] = labeled["Title"] + " [SEP] " + labeled["Abstract"]
labeled["view_2"] = labeled["Abstract"] + " [SEP] " + labeled["Title"]

unlabeled["view_1"] = unlabeled["Title"] + " [SEP] " + unlabeled["Abstract"]
unlabeled["view_2"] = unlabeled["Abstract"] + " [SEP] " + unlabeled["Title"]

In [ ]:
train_df, test_df = train_test_split(
    labeled,
    test_size=0.35,
    stratify=labeled["Label"],
    random_state=42
)

train_sub, val_sub = train_test_split(
    train_df,
    test_size=0.25,
    stratify=train_df["Label"],
    random_state=42
)

print("Train:",train_sub.shape)
print("Validation:",val_sub.shape)
print("Test:",test_df.shape)


In [ ]:
MODEL_MAP = {
    "bert": "bert-base-uncased",
    "scibert": "allenai/scibert_scivocab_uncased",
    "biobert": "dmis-lab/biobert-base-cased-v1.2",
    "pubmedbert": "microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract",
    "biolinkbert": "michiyasunaga/BioLinkBERT-base"
}

MODEL_1_NAME = "bert"
MODEL_2_NAME = "scibert"

MODEL_1 = MODEL_MAP[MODEL_1_NAME]
MODEL_2 = MODEL_MAP[MODEL_2_NAME]

In [ ]:
tok_model1 = AutoTokenizer.from_pretrained(MODEL_1)
tok_model2 = AutoTokenizer.from_pretrained(MODEL_2)

In [ ]:
MAX_LEN = 256
BATCH_SIZE = 16

MLM_EPOCHS = 15

ITERATIONS = 3

EPOCH_LIST = [25,20,15]

MAX_PSEUDO_LIST = [100,150,200]

THRESHOLDS = [0.95,0.92,0.90]

CONSISTENCY_WEIGHT = 0.2

MC_DROPOUT_PASSES = 8
UNCERTAINTY_THRESHOLD = 0.05

SEEDS = [42, 123, 2023, 777, 999]

LR_LIST = [1e-5, 2e-5, 3e-5, 5e-5]

results = []

In [ ]:
def encode(df, tokenizer, col):

    enc = tokenizer(
        df[col].tolist(),
        padding="max_length",
        truncation=True,
        max_length=MAX_LEN,
        return_tensors="pt"
    )

    labels = torch.tensor(df["Label"].values)

    if "confidence" in df.columns:
        confidence = torch.tensor(df["confidence"].values, dtype=torch.float)
    else:
        confidence = torch.ones(len(df))

    return TensorDataset(
        enc["input_ids"],
        enc["attention_mask"],
        labels,
        confidence
    )


def encode_unlabeled(df, tokenizer, col):

    enc = tokenizer(
        df[col].tolist(),
        padding="max_length",
        truncation=True,
        max_length=MAX_LEN,
        return_tensors="pt"
    )

    return TensorDataset(enc["input_ids"], enc["attention_mask"])

In [ ]:
def train_mlm(model_name, tokenizer, column):

    model = AutoModelForMaskedLM.from_pretrained(model_name).to(DEVICE)

    df_all = pd.concat([train_df, unlabeled])

    enc = tokenizer(
        df_all[column].tolist(),
        padding="max_length",
        truncation=True,
        max_length=MAX_LEN,
        return_tensors="pt"
    )

    dataset = [
        {"input_ids":i,"attention_mask":m}
        for i,m in zip(enc["input_ids"],enc["attention_mask"])
    ]

    loader = DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        collate_fn=DataCollatorForLanguageModeling(
            tokenizer,
            mlm_probability=0.15
        )
    )

    optimizer = AdamW(model.parameters(), lr=5e-5)

    for _ in range(MLM_EPOCHS):

        for batch in loader:

            batch = {k:v.to(DEVICE) for k,v in batch.items()}

            loss = model(**batch).loss

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

    return model.base_model.state_dict()

In [ ]:
def train_model(model, tokenizer, df, col, epochs, lr):

    loader = DataLoader(
        encode(df, tokenizer, col),
        batch_size=BATCH_SIZE,
        shuffle=True
    )

    optimizer = AdamW(model.parameters(), lr=lr)

    if USE_LR_SCHEDULER:
        total_steps = len(loader) * epochs
        warmup_steps = int(0.1 * total_steps)
        scheduler = get_linear_schedule_with_warmup(
            optimizer,
            num_warmup_steps=warmup_steps,
            num_training_steps=total_steps
        )

    scaler = torch.cuda.amp.GradScaler() if USE_MIXED_PRECISION else None

    model.train()

    for epoch in range(epochs):

        for batch in loader:

            input_ids, attention_mask, labels, confidence = [b.to(DEVICE) for b in batch]
            optimizer.zero_grad()

            if USE_MIXED_PRECISION:
                with torch.cuda.amp.autocast():
                    logits = model(
                        input_ids=input_ids,
                        attention_mask=attention_mask
                    ).logits
                    sup_loss = F.cross_entropy(logits, labels)
                    logits2 = model(
                        input_ids=input_ids,
                        attention_mask=attention_mask
                    ).logits
                    p1 = torch.softmax(logits, dim=1)
                    p2 = torch.softmax(logits2, dim=1)
                    cons_loss = F.mse_loss(p1.detach(), p2)
                    loss = sup_loss + CONSISTENCY_WEIGHT * cons_loss
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            else:
                logits = model(
                    input_ids=input_ids,
                    attention_mask=attention_mask
                ).logits
                sup_loss = F.cross_entropy(logits, labels)
                logits2 = model(
                    input_ids=input_ids,
                    attention_mask=attention_mask
                ).logits
                p1 = torch.softmax(logits, dim=1)
                p2 = torch.softmax(logits2, dim=1)
                cons_loss = F.mse_loss(p1.detach(), p2)
                loss = sup_loss + CONSISTENCY_WEIGHT * cons_loss
                loss.backward()
                optimizer.step()

            if USE_LR_SCHEDULER:
                scheduler.step()

    return model

In [ ]:

mlm_model1 = train_mlm(MODEL_1, tok_model1, "view_1")
mlm_model2 = train_mlm(MODEL_2, tok_model2, "view_2")



In [ ]:
def mc_dropout_predict(model,input_ids,attention_mask):

    model.train()

    preds=[]

    with torch.no_grad():

        for _ in range(MC_DROPOUT_PASSES):

            logits=model(
                input_ids=input_ids,
                attention_mask=attention_mask
            ).logits

            preds.append(torch.softmax(logits,dim=1))

    preds=torch.stack(preds)

    mean=preds.mean(0)
    var=preds.var(0).mean(dim=1)

    return mean,var

In [ ]:
def generate_pseudo(model1, model2, tok1, tok2, view1, view2, threshold, max_pseudo):

    loader1 = DataLoader(
        encode_unlabeled(unlabeled, tok1, view1),
        batch_size=8,
        shuffle=False
    )

    loader2 = DataLoader(
        encode_unlabeled(unlabeled, tok2, view2),
        batch_size=8,
        shuffle=False
    )

    pseudo_rows = []

    with torch.no_grad():

        for i, (b1, b2) in enumerate(zip(loader1, loader2)):

            b1 = [x.to(DEVICE) for x in b1]
            b2 = [x.to(DEVICE) for x in b2]

            p1, var1 = mc_dropout_predict(model1, b1[0], b1[1])
            p2, var2 = mc_dropout_predict(model2, b2[0], b2[1])

            conf1, pred1 = torch.max(p1, dim=1)
            conf2, pred2 = torch.max(p2, dim=1)

            agree = (pred1 == pred2)
            confident = (conf1 >= threshold) & (conf2 >= threshold)
            stable = (var1 < UNCERTAINTY_THRESHOLD) & (var2 < UNCERTAINTY_THRESHOLD)

            mask = agree & confident & stable

            if mask.sum() == 0:
                continue


            start_idx = i * loader1.batch_size
            idx = torch.arange(start_idx, start_idx + len(pred1), device=pred1.device)[mask]

            idx = idx.cpu().numpy()

            selected = unlabeled.iloc[idx].copy()
            selected["Label"] = pred1[mask].cpu().numpy()

            pseudo_rows.append(selected)

            del b1, b2, p1, p2, var1, var2
            torch.cuda.empty_cache()

    if len(pseudo_rows) == 0:
        return None, None

    pseudo_df = pd.concat(pseudo_rows)

    balanced = []

    for c in [0, 1]:
        subset = pseudo_df[pseudo_df["Label"] == c]
        balanced.append(subset.head(max_pseudo))

    pseudo_df = pd.concat(balanced)

    return pseudo_df, pseudo_df.copy()

In [ ]:
def evaluate(model, tokenizer, df, col):

    loader = DataLoader(
        encode(df, tokenizer, col),
        batch_size=BATCH_SIZE
    )

    probs=[]
    gold=[]

    model.eval()

    with torch.no_grad():

        for batch in loader:

            batch=[b.to(DEVICE) for b in batch]

            logits=model(
                input_ids=batch[0],
                attention_mask=batch[1]
            ).logits

            p=torch.softmax(logits,dim=1)

            probs.extend(p[:,1].cpu().numpy())
            gold.extend(batch[2].cpu().numpy())

    probs=np.array(probs)
    gold=np.array(gold)

    fpr,tpr,thr=roc_curve(gold,probs)
    best_thr=thr[np.argmax(tpr-fpr)]

    preds=(probs>=best_thr).astype(int)

    print(classification_report(gold,preds))

    return probs,gold

In [ ]:


best_accuracy = -1
best_f1 = -1


best_model1 = None
best_model2 = None

best_lr = None
best_seed = None
best_weights = None
best_threshold = None

In [ ]:
import copy

for LR in LR_LIST:

    print("\n")
    print("LEARNING RATE:", LR)
    print("\n")

    for seed in SEEDS:

        print("\nSeed:", seed)

        set_seed(seed)

        model1 = AutoModelForSequenceClassification.from_pretrained(
            MODEL_1,
            num_labels=2
        ).to(DEVICE)

        model2 = AutoModelForSequenceClassification.from_pretrained(
            MODEL_2,
            num_labels=2
        ).to(DEVICE)

        model1.base_model.load_state_dict(mlm_model1, strict=False)
        model2.base_model.load_state_dict(mlm_model2, strict=False)

        train_1 = train_sub.copy()
        train_2 = train_sub.copy()

        for it in range(ITERATIONS):

            print("\n======================")
            print("ITERATION", it+1)
            print("======================")

            thr = THRESHOLDS[it]
            max_pseudo = MAX_PSEUDO_LIST[it]
            epochs = EPOCH_LIST[it]

            model1 = train_model(model1, tok_model1, train_1, "view_1", epochs, LR)
            model2 = train_model(model2, tok_model2, train_2, "view_2", epochs, LR)

            torch.cuda.empty_cache()
            gc.collect()

            pseudo_1, pseudo_2 = generate_pseudo(
                model1,
                model2,
                tok_model1,
                tok_model2,
                "view_1",
                "view_2",
                thr,
                max_pseudo
            )

            if pseudo_1 is not None:

                print("Pseudo samples:", len(pseudo_1))


                train_2 = pd.concat([train_2, pseudo_1], ignore_index=True)
                train_1 = pd.concat([train_1, pseudo_2], ignore_index=True)

            else:
                print("No pseudo labels generated")


            print(f"\n{MODEL_1_NAME.upper()} TEST")

            prob_1, gold = evaluate(
                model1,
                tok_model1,
                test_df,
                "view_1"
            )

            fpr_1, tpr_1, thr_1 = roc_curve(gold, prob_1)
            best_thr_1 = thr_1[np.argmax(tpr_1 - fpr_1)]

            pred_1 = (prob_1 >= best_thr_1).astype(int)
            f1_1 = f1_score(gold, pred_1, average="weighted")

            print(f"{MODEL_1_NAME} F1:", f1_1)


            print(f"\n{MODEL_2_NAME.upper()} TEST")

            prob_2, _ = evaluate(
                model2,
                tok_model2,
                test_df,
                "view_2"
            )

            fpr_2, tpr_2, thr_2 = roc_curve(gold, prob_2)
            best_thr_2 = thr_2[np.argmax(tpr_2 - fpr_2)]

            pred_2 = (prob_2 >= best_thr_2).astype(int)
            f1_2 = f1_score(gold, pred_2, average="weighted")

            print(f"{MODEL_2_NAME} F1:", f1_2)


            total = f1_1 + f1_2 + 1e-8

            w_1 = f1_1 / total
            w_2 = f1_2 / total

            print("Adaptive weights:", round(w_1, 3), round(w_2, 3))

            final_prob = w_1 * prob_1 + w_2 * prob_2

            fpr, tpr, thr = roc_curve(gold, final_prob)
            best_thr = thr[np.argmax(tpr - fpr)]

            pred = (final_prob >= best_thr).astype(int)

            print("\nENSEMBLE RESULTS")
            print(classification_report(gold, pred))

            acc = accuracy_score(gold, pred)
            f1 = f1_score(gold, pred, average="weighted")

            print("Accuracy:", acc)
            print("F1:", f1)

            gc.collect()
            torch.cuda.empty_cache()

        results.append({
            "lr": LR,
            "seed": seed,
            "accuracy": acc,
            "f1": f1
        })


        if f1 > best_f1:

            best_f1 = f1
            best_accuracy = acc

            best_lr = LR
            best_seed = seed

            best_model1 = copy.deepcopy(model1).cpu()
            best_model2 = copy.deepcopy(model2).cpu()

            best_weights = (w_1, w_2)
            best_threshold = best_thr

            print("\nNEW BEST MODEL SAVED")
            print("LR:", best_lr)
            print("Seed:", best_seed)
            print("Best F1:", best_f1)

In [ ]:
print("\n")
print("USING BEST ENSEMBLE MODEL")
print("\n")

print("Best LR:", best_lr)
print("Best Seed:", best_seed)
print("Best Accuracy:", best_accuracy)
print("Best F1:", best_f1)

w_1, w_2 = best_weights

print("Best ensemble weights:", w_1, w_2)

best_model1 = best_model1.to(DEVICE)
best_model2 = best_model2.to(DEVICE)

loader1 = DataLoader(
    encode_unlabeled(unlabeled, tok_model1, "view_1"),
    batch_size=16,
    shuffle=False
)

loader2 = DataLoader(
    encode_unlabeled(unlabeled, tok_model2, "view_2"),
    batch_size=16,
    shuffle=False
)

probs_1 = []
probs_2 = []

best_model1.eval()
best_model2.eval()

with torch.no_grad():

    for (b1, b2) in tqdm(zip(loader1, loader2), total=len(loader1)):

        b1 = [x.to(DEVICE) for x in b1]
        b2 = [x.to(DEVICE) for x in b2]

        logits_1 = best_model1(
            input_ids=b1[0],
            attention_mask=b1[1]
        ).logits

        logits_2 = best_model2(
            input_ids=b2[0],
            attention_mask=b2[1]
        ).logits

        prob_1 = torch.softmax(logits_1, dim=1)[:, 1]
        prob_2 = torch.softmax(logits_2, dim=1)[:, 1]

        probs_1.extend(prob_1.cpu().numpy())
        probs_2.extend(prob_2.cpu().numpy())

probs_1 = np.array(probs_1)
probs_2 = np.array(probs_2)

ensemble_probs = w_1 * probs_1 + w_2 * probs_2

pseudo_labels = (ensemble_probs >= best_threshold).astype(int)

In [ ]:
pseudo_df = unlabeled.copy()

pseudo_df[f"{MODEL_1_NAME}_prob"] = probs_1
pseudo_df[f"{MODEL_2_NAME}_prob"] = probs_2
pseudo_df["Ensemble_prob"] = ensemble_probs
pseudo_df["Pseudo_Label"] = pseudo_labels

save_path = f"/content/drive/MyDrive/eq_5d/dataset/co-training/{MODEL_1_NAME}_{MODEL_2_NAME}_best_ensemble_pseudo_labels.csv"

pseudo_df.to_csv(save_path, index=False)

print("\nPseudo labels saved to:")
print(save_path)

print("\nPseudo label distribution:")
print(pseudo_df["Pseudo_Label"].value_counts())

In [ ]:
results_df = pd.DataFrame(results)

print("\nFULL RESULTS")
print(results_df)

summary = results_df.groupby(["lr"]).agg({
    "accuracy": ["mean", "std"],
    "f1": ["mean", "std"]
}).reset_index()

print("\nSUMMARY BY LEARNING RATE")
print(summary)

In [ ]:

def compute_ci(values):

    mean = np.mean(values)
    std = np.std(values, ddof=1)
    n = len(values)

    if n < 2:
        return mean, 0.0, mean, mean

    ci = 1.96 * std / np.sqrt(n)

    lower = mean - ci
    upper = mean + ci

    return mean, std, lower, upper

In [ ]:
for lr in LR_LIST:

    subset = results_df[results_df["lr"] == lr]

    if len(subset) == 0:
        continue

    f1_mean, f1_std, f1_low, f1_high = compute_ci(subset["f1"])
    acc_mean, acc_std, acc_low, acc_high = compute_ci(subset["accuracy"])

    print("\n")
    print("Learning Rate:", lr)
    print("==============================")

    print(f"Mean F1 ± Std: {f1_mean:.4f} ± {f1_std:.4f}")
    print(f"95% CI F1: [{f1_low:.4f}, {f1_high:.4f}]")

    print(f"Mean Accuracy ± Std: {acc_mean:.4f} ± {acc_std:.4f}")
    print(f"95% CI Accuracy: [{acc_low:.4f}, {acc_high:.4f}]")